<a href="https://colab.research.google.com/github/kasrasa/Object-detection-tutorial/blob/Faster-RCNN/Faster_RCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Faster R-CNN Small-Object Experiments

1. Setup and configuration  
2. Shared utilities  
3. COCO dataset loading and small-object subset creation  
4. Faster R-CNN / RPN inspection  
5. RPN recall evaluation  
6. Fine-tuning default and small-anchor Faster R-CNN  
7. Per-class small-object evaluation  
8. Hard-example mining from the full COCO training annotations  
9. Expanded-data training and before/after comparison  


In [ ]:
!pip install -q pycocotools


In [ ]:
import os
import random
import urllib.request
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import box_iou
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.rpn import AnchorGenerator

from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from pycocotools.coco import COCO
from IPython.display import display

print("Torch version:", torch.__version__)
print("TorchVision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
# -------------------------
# Global configuration
# -------------------------

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# COCO annotation paths uploaded in Colab.
TRAIN_ANN = "/content/data/train/instances_train2014.json"
VAL_ANN = "/content/data/valid/instances_val2014.json"

# Image roots. Images are downloaded on demand into these folders.
TRAIN_IMAGE_ROOT = Path("/content/data/images/train2014")
VAL_IMAGE_ROOT = Path("/content/data/images/val2014")
TRAIN_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
VAL_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

# Dataset sizes.
NUM_DEBUG_IMAGES = 30
NUM_TRAIN = 200
NUM_VAL = 50
MIN_SMALL_OBJECTS_PER_IMAGE = 3

# Training settings.
BATCH_SIZE = 1
NUM_WORKERS = 2
NUM_EPOCHS = 10
LR = 0.001
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005

# Evaluation / mining settings.
IOU_THRESH = 0.5
SCORE_THRESH = 0.05
POOR_RECALL_THRESHOLD = 0.7
MIN_SMALL_GT = 3
NUM_ADDED_HARD_IMAGES = 100
SUSPECT_ID = 8  # Example: truck in COCO category ids. Change for class-level debugging.

# RPN proposal settings used for small-object experiments.
RPN_PROPOSAL_SETTINGS = {
    "pre_nms_top_n_train": 4000,
    "post_nms_top_n_train": 2000,
    "pre_nms_top_n_test": 2000,
    "post_nms_top_n_test": 1000,
}

# Anchor setup used for small-anchor experiment.
SMALL_ANCHOR_SIZES = ((8,), (16,), (32,), (64,), (128,))
ANCHOR_ASPECT_RATIOS = ((0.5, 1.0, 2.0),) * 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Using device:", DEVICE)
print("Train annotations exist:", os.path.exists(TRAIN_ANN))
print("Val annotations exist:", os.path.exists(VAL_ANN))


## 1. Shared utilities


In [ ]:
def print_tensor_info(name, x):
    """Print a compact summary of a tensor."""
    if isinstance(x, torch.Tensor):
        print(f"{name}")
        print(f"  shape: {tuple(x.shape)}")
        print(f"  dtype: {x.dtype}")
        print(f"  device: {x.device}")
        if x.numel() > 0:
            print(f"  min/max: {x.min().item():.4f} / {x.max().item():.4f}")
    else:
        print(f"{name}: {type(x)}")


def to_device_targets(targets, device):
    """Move a list of target dictionaries to device."""
    return [
        {k: v.to(device) if torch.is_tensor(v) else v for k, v in target.items()}
        for target in targets
    ]


def xywh_to_xyxy(box):
    """COCO [x, y, w, h] -> TorchVision [x1, y1, x2, y2]."""
    x, y, w, h = box
    return [x, y, x + w, y + h]


def box_area_xyxy(box):
    """Area of one xyxy box."""
    x1, y1, x2, y2 = box
    return max(0.0, float(x2 - x1)) * max(0.0, float(y2 - y1))


def coco_size_bucket_from_area(area):
    """COCO-style small/medium/large area bucket."""
    area = float(area)
    if area < 32 ** 2:
        return "small"
    if area < 96 ** 2:
        return "medium"
    return "large"


def get_size_mask(areas, size_bucket):
    """Return boolean mask for areas in a COCO size bucket."""
    if size_bucket == "small":
        return areas < 32 ** 2
    if size_bucket == "medium":
        return (areas >= 32 ** 2) & (areas < 96 ** 2)
    if size_bucket == "large":
        return areas >= 96 ** 2
    if size_bucket == "all":
        return torch.ones_like(areas, dtype=torch.bool)
    raise ValueError(f"Unknown size_bucket: {size_bucket}")


def valid_detection_ann(ann):
    """Filter invalid/crowd annotations for object detection training/evaluation."""
    if ann.get("iscrowd", 0) == 1:
        return False

    x, y, w, h = ann["bbox"]
    if w <= 1 or h <= 1:
        return False

    if ann.get("area", w * h) <= 1:
        return False

    return True


def get_cat_name(coco, cat_id):
    """Return COCO category name for a category id."""
    cat_id = int(cat_id)
    if cat_id in coco.cats:
        return coco.cats[cat_id]["name"]
    return f"cat_id={cat_id}"


def show_image_with_boxes(
    image,
    boxes=None,
    labels=None,
    scores=None,
    categories=None,
    max_boxes=20,
    title=None,
    box_color="red",
):
    """
    Display a PIL image or Tensor[C,H,W] with xyxy boxes.
    This replaces the previous separate PIL/tensor visualization helpers.
    """
    if isinstance(image, torch.Tensor):
        image_to_show = image.detach().cpu().permute(1, 2, 0).numpy()
    else:
        image_to_show = image

    fig, ax = plt.subplots(1, figsize=(12, 9))
    ax.imshow(image_to_show)

    if boxes is not None:
        boxes = boxes.detach().cpu() if torch.is_tensor(boxes) else torch.as_tensor(boxes)
        n = min(len(boxes), max_boxes)

        for i in range(n):
            x1, y1, x2, y2 = boxes[i].tolist()
            rect = patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2,
                edgecolor=box_color,
                facecolor="none",
            )
            ax.add_patch(rect)

            text = ""
            if labels is not None:
                label_id = int(labels[i])
                text += categories[label_id] if categories is not None and label_id < len(categories) else str(label_id)

            if scores is not None:
                text += f" {float(scores[i]):.2f}"

            if text:
                ax.text(
                    x1,
                    y1,
                    text,
                    bbox=dict(facecolor="white", alpha=0.7),
                    fontsize=8,
                )

    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.show()


In [ ]:
def download_coco2014_image(img_info, split, image_root):
    """Download one COCO 2014 image if it is not already present."""
    image_root = Path(image_root)
    image_root.mkdir(parents=True, exist_ok=True)

    file_name = img_info["file_name"]
    out_path = image_root / file_name

    if out_path.exists():
        return out_path

    url = f"http://images.cocodataset.org/{split}/{file_name}"

    try:
        urllib.request.urlretrieve(url, out_path)
        return out_path
    except Exception as e:
        print("Failed:", url)
        print(e)
        return None


def image_small_object_stats(coco, img_id):
    """Return valid annotations grouped by COCO object size bucket."""
    ann_ids = coco.getAnnIds(imgIds=[img_id], iscrowd=False)
    anns = coco.loadAnns(ann_ids)
    anns = [ann for ann in anns if valid_detection_ann(ann)]

    stats = {"all": anns, "small": [], "medium": [], "large": []}

    for ann in anns:
        stats[coco_size_bucket_from_area(ann["area"])].append(ann)

    return stats


def find_images_with_small_objects(coco, min_small_objects=3):
    """Find image ids containing at least min_small_objects valid small objects."""
    img_ids = []

    for img_id in coco.imgs.keys():
        stats = image_small_object_stats(coco, img_id)
        if len(stats["small"]) >= min_small_objects:
            img_ids.append(img_id)

    return img_ids


def download_image_subset(coco, img_ids, split, image_root, max_print=5):
    """Download a list of COCO images."""
    paths = []

    for i, img_id in enumerate(img_ids):
        img_info = coco.loadImgs(img_id)[0]
        path = download_coco2014_image(img_info, split=split, image_root=image_root)
        paths.append(path)

        if i < max_print:
            print(i, img_info["file_name"], path)

    return paths


def show_coco_image_with_gt(coco, img_id, image_root, max_boxes=50):
    """Display a COCO image with GT boxes colored by size bucket."""
    img_info = coco.loadImgs(img_id)[0]
    img_path = Path(image_root) / img_info["file_name"]
    image = Image.open(img_path).convert("RGB")

    stats = image_small_object_stats(coco, img_id)
    anns = stats["all"][:max_boxes]

    fig, ax = plt.subplots(1, figsize=(12, 9))
    ax.imshow(image)

    for ann in anns:
        x1, y1, x2, y2 = xywh_to_xyxy(ann["bbox"])
        bucket = coco_size_bucket_from_area(ann["area"])
        cat_name = get_cat_name(coco, ann["category_id"])

        edge = {"small": "red", "medium": "orange", "large": "lime"}[bucket]

        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor=edge,
            facecolor="none",
        )
        ax.add_patch(rect)

        ax.text(
            x1,
            y1,
            f"{cat_name} | {bucket}",
            bbox=dict(facecolor="white", alpha=0.7),
            fontsize=8,
        )

    ax.set_title(
        f"{img_info['file_name']} | "
        f"small={len(stats['small'])}, medium={len(stats['medium'])}, large={len(stats['large'])}"
    )
    ax.axis("off")
    plt.show()


In [ ]:
class CocoDetectionForFasterRCNN(Dataset):
    """COCO detection dataset returning TorchVision Faster R-CNN style targets."""

    def __init__(self, coco, img_ids, image_root):
        self.coco = coco
        self.img_ids = list(img_ids)
        self.image_root = Path(image_root)

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = self.image_root / img_info["file_name"]

        image = Image.open(img_path).convert("RGB")
        image_tensor = torchvision.transforms.functional.to_tensor(image)

        ann_ids = self.coco.getAnnIds(imgIds=[img_id], iscrowd=False)
        anns = self.coco.loadAnns(ann_ids)
        anns = [ann for ann in anns if valid_detection_ann(ann)]

        boxes, labels, areas, iscrowd = [], [], [], []

        for ann in anns:
            boxes.append(xywh_to_xyxy(ann["bbox"]))

            # COCO category_id is not contiguous from 1 to 80.
            # TorchVision COCO-pretrained Faster R-CNN uses COCO-style category ids.
            labels.append(ann["category_id"])
            areas.append(ann["area"])
            iscrowd.append(ann.get("iscrowd", 0))

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([img_id], dtype=torch.int64),
            "area": torch.as_tensor(areas, dtype=torch.float32),
            "iscrowd": torch.as_tensor(iscrowd, dtype=torch.int64),
        }

        return image_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


def make_loader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate_fn,
        num_workers=num_workers,
    )


In [ ]:
def apply_rpn_proposal_settings(model, settings=RPN_PROPOSAL_SETTINGS):
    """Apply centralized RPN pre/post NMS proposal settings."""
    model.rpn.pre_nms_top_n_train = settings["pre_nms_top_n_train"]
    model.rpn.post_nms_top_n_train = settings["post_nms_top_n_train"]
    model.rpn.pre_nms_top_n_test = settings["pre_nms_top_n_test"]
    model.rpn.post_nms_top_n_test = settings["post_nms_top_n_test"]
    return model


def make_small_anchor_generator():
    """Create the small-anchor generator used in the small-object experiment."""
    return AnchorGenerator(
        sizes=SMALL_ANCHOR_SIZES,
        aspect_ratios=ANCHOR_ASPECT_RATIOS,
    )


def make_fasterrcnn_model(
    use_small_anchors=False,
    freeze_backbone=False,
    apply_proposal_settings=False,
    device=DEVICE,
):
    """Create a COCO-pretrained Faster R-CNN model with optional small anchors."""
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT

    kwargs = {"weights": weights}
    if use_small_anchors:
        kwargs["rpn_anchor_generator"] = make_small_anchor_generator()

    model = fasterrcnn_resnet50_fpn(**kwargs).to(device)

    if freeze_backbone:
        for param in model.backbone.parameters():
            param.requires_grad = False

    if apply_proposal_settings:
        apply_rpn_proposal_settings(model)

    categories = weights.meta["categories"]
    return model, categories


def make_optimizer(model):
    return torch.optim.SGD(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )


def make_scaler():
    return torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())


In [ ]:
def proposal_recall_at_iou(gt_boxes, proposal_boxes, iou_threshold=0.5):
    """Fraction of GT boxes covered by at least one proposal at an IoU threshold."""
    if len(gt_boxes) == 0:
        return None, None

    ious = box_iou(gt_boxes, proposal_boxes)
    max_iou_per_gt = ious.max(dim=1).values
    matched = max_iou_per_gt >= iou_threshold
    recall = matched.float().mean().item()

    return recall, max_iou_per_gt


def run_internal_frcnn_pipeline(model, images, targets=None, device=DEVICE):
    """
    Run transform -> backbone -> RPN and return intermediate outputs.
    images: list[Tensor[C,H,W]]
    targets: optional list[dict]
    """
    model.eval()

    images_device = [img.to(device) for img in images]
    targets_device = to_device_targets(targets, device) if targets is not None else None

    with torch.no_grad():
        transformed_images, transformed_targets = model.transform(images_device, targets_device)
        features = model.backbone(transformed_images.tensors)
        proposals, rpn_losses = model.rpn(transformed_images, features, transformed_targets)

    return {
        "transformed_images": transformed_images,
        "transformed_targets": transformed_targets,
        "features": features,
        "proposals": proposals,
        "rpn_losses": rpn_losses,
    }


def recall_by_size_bucket(original_target, transformed_target, proposals, iou_threshold=0.5):
    """Compute RPN proposal recall by COCO size bucket for one image."""
    original_areas = original_target["area"].to(proposals.device)
    gt_boxes = transformed_target["boxes"]

    result = {}

    for bucket in ["small", "medium", "large"]:
        mask = get_size_mask(original_areas, bucket)
        bucket_gt_boxes = gt_boxes[mask]

        if len(bucket_gt_boxes) == 0:
            result[bucket] = {"num_gt": 0, "recall": None, "max_iou_per_gt": None}
            continue

        recall, max_iou_per_gt = proposal_recall_at_iou(
            bucket_gt_boxes,
            proposals,
            iou_threshold=iou_threshold,
        )

        result[bucket] = {
            "num_gt": len(bucket_gt_boxes),
            "recall": recall,
            "max_iou_per_gt": max_iou_per_gt.detach().cpu(),
        }

    return result


def evaluate_rpn_recall_dataset(model, dataloader, device=DEVICE, iou_threshold=0.5, max_batches=None):
    """Evaluate RPN proposal recall by size bucket across a dataloader."""
    model.eval()

    totals = {
        "small": {"matched": 0, "total": 0},
        "medium": {"matched": 0, "total": 0},
        "large": {"matched": 0, "total": 0},
        "all": {"matched": 0, "total": 0},
    }
    per_image_rows = []

    with torch.no_grad():
        for batch_idx, (images, targets) in enumerate(dataloader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            pipe = run_internal_frcnn_pipeline(model, images, targets, device=device)
            transformed_targets = pipe["transformed_targets"]
            proposals = pipe["proposals"]

            for i in range(len(images)):
                gt_boxes = transformed_targets[i]["boxes"]
                prop_boxes = proposals[i]
                original_areas = targets[i]["area"].to(device)

                if len(gt_boxes) == 0:
                    continue

                ious = box_iou(gt_boxes, prop_boxes)
                max_iou_per_gt = ious.max(dim=1).values
                matched = max_iou_per_gt >= iou_threshold

                totals["all"]["matched"] += int(matched.sum().item())
                totals["all"]["total"] += len(gt_boxes)

                row = {
                    "image_id": int(targets[i]["image_id"].item()),
                    "all_total": len(gt_boxes),
                    "all_matched": int(matched.sum().item()),
                }

                for bucket in ["small", "medium", "large"]:
                    mask = get_size_mask(original_areas, bucket)
                    bucket_total = int(mask.sum().item())
                    bucket_matched = int((matched & mask).sum().item())

                    totals[bucket]["matched"] += bucket_matched
                    totals[bucket]["total"] += bucket_total

                    row[f"{bucket}_total"] = bucket_total
                    row[f"{bucket}_matched"] = bucket_matched

                per_image_rows.append(row)

    summary = {}
    for bucket, values in totals.items():
        total = values["total"]
        matched = values["matched"]
        summary[bucket] = {
            "matched": matched,
            "total": total,
            "recall": matched / total if total > 0 else None,
        }

    return summary, per_image_rows


In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, device=DEVICE, epoch=0, print_every=25):
    """One Faster R-CNN training epoch with AMP support."""
    model.train()
    running_loss = 0.0

    for step, (images, targets) in enumerate(loader):
        images = [img.to(device) for img in images]
        targets = to_device_targets(targets, device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += float(loss.detach().cpu())

        if step % print_every == 0:
            loss_text = " | ".join(
                f"{name}: {float(value.detach().cpu()):.3f}"
                for name, value in loss_dict.items()
            )
            print(
                f"epoch {epoch} step {step}/{len(loader)} "
                f"total={float(loss.detach().cpu()):.3f} | {loss_text}"
            )

    return running_loss / max(1, len(loader))


def train_fixed_steps(model, loader, optimizer, scaler, device=DEVICE, max_steps=400, print_every=25):
    """Train for a fixed number of optimizer steps to control comparisons."""
    model.train()
    step_count = 0
    running_loss = 0.0

    while step_count < max_steps:
        for images, targets in loader:
            if step_count >= max_steps:
                break

            images = [img.to(device) for img in images]
            targets = to_device_targets(targets, device)

            optimizer.zero_grad()

            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                loss_dict = model(images, targets)
                loss = sum(loss_dict.values())

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += float(loss.detach().cpu())

            if step_count % print_every == 0:
                print(f"step {step_count}/{max_steps} loss={float(loss.detach().cpu()):.4f}")

            step_count += 1

    return running_loss / max_steps


In [ ]:
def match_predictions_to_gt(output, target, iou_threshold=0.5, score_threshold=0.05):
    """
    One-to-one GT-centered matching.

    Rules:
    - Match only same-class predictions to GT.
    - Predictions must have score >= score_threshold.
    - Prediction-GT IoU must be >= iou_threshold.
    - Each GT can be matched at most once.
    - Each prediction can be matched at most once.
    """
    gt_boxes = target["boxes"].detach().cpu()
    gt_labels = target["labels"].detach().cpu()
    gt_areas = target["area"].detach().cpu()

    pred_boxes = output["boxes"].detach().cpu()
    pred_labels = output["labels"].detach().cpu()
    pred_scores = output["scores"].detach().cpu()

    keep = pred_scores >= score_threshold
    pred_boxes = pred_boxes[keep]
    pred_labels = pred_labels[keep]
    pred_scores = pred_scores[keep]

    num_gt = len(gt_boxes)
    matched = torch.zeros(num_gt, dtype=torch.bool)
    matched_iou = torch.zeros(num_gt, dtype=torch.float32)
    matched_score = torch.zeros(num_gt, dtype=torch.float32)

    candidates = []

    for gt_idx in range(num_gt):
        same_class = pred_labels == gt_labels[gt_idx]
        if same_class.sum() == 0:
            continue

        candidate_pred_indices = torch.where(same_class)[0]
        ious = box_iou(gt_boxes[gt_idx : gt_idx + 1], pred_boxes[candidate_pred_indices])[0]

        for local_idx, pred_idx in enumerate(candidate_pred_indices):
            iou = float(ious[local_idx])
            if iou >= iou_threshold:
                candidates.append(
                    {
                        "gt_idx": gt_idx,
                        "pred_idx": int(pred_idx),
                        "iou": iou,
                        "score": float(pred_scores[pred_idx]),
                    }
                )

    # Greedy matching: highest score first, then highest IoU.
    candidates = sorted(candidates, key=lambda x: (x["score"], x["iou"]), reverse=True)

    used_gt = set()
    used_pred = set()

    for cand in candidates:
        if cand["gt_idx"] in used_gt or cand["pred_idx"] in used_pred:
            continue

        used_gt.add(cand["gt_idx"])
        used_pred.add(cand["pred_idx"])

        matched[cand["gt_idx"]] = True
        matched_iou[cand["gt_idx"]] = cand["iou"]
        matched_score[cand["gt_idx"]] = cand["score"]

    rows = []
    for gt_idx in range(num_gt):
        area = float(gt_areas[gt_idx])
        size_bucket = coco_size_bucket_from_area(area)

        if matched[gt_idx]:
            reason = "matched"
        else:
            # Useful diagnostics for why the GT was missed.
            same_class = pred_labels == gt_labels[gt_idx]
            reason = "no_same_class_prediction" if same_class.sum() == 0 else "same_class_low_iou"

        rows.append(
            {
                "gt_idx": gt_idx,
                "category_id": int(gt_labels[gt_idx]),
                "size_bucket": size_bucket,
                "matched": bool(matched[gt_idx]),
                "best_iou": float(matched_iou[gt_idx]),
                "best_score": float(matched_score[gt_idx]),
                "reason": reason,
            }
        )

    return rows


def evaluate_gt_centered_per_class(
    model,
    dataloader,
    coco,
    device=DEVICE,
    iou_threshold=0.5,
    score_threshold=0.05,
):
    """Run GT-centered detection evaluation and return one row per GT object."""
    model.eval()
    all_rows = []

    with torch.no_grad():
        for images, targets in dataloader:
            outputs = model([img.to(device) for img in images])

            for target, output in zip(targets, outputs):
                image_id = int(target["image_id"].item())

                rows = match_predictions_to_gt(
                    output=output,
                    target=target,
                    iou_threshold=iou_threshold,
                    score_threshold=score_threshold,
                )

                for row in rows:
                    row["image_id"] = image_id
                    row["category_name"] = get_cat_name(coco, row["category_id"])

                all_rows.extend(rows)

    return pd.DataFrame(all_rows)


def summarize_small_object_class_performance(df, min_gt=3):
    """Summarize detection performance for small objects by class."""
    small_df = df[df["size_bucket"] == "small"].copy()

    summary = (
        small_df.groupby(["category_id", "category_name"])
        .agg(
            num_small_gt=("matched", "size"),
            num_matched=("matched", "sum"),
            recall=("matched", "mean"),
            avg_best_iou=("best_iou", "mean"),
            avg_best_score=("best_score", "mean"),
        )
        .reset_index()
    )

    summary = summary[summary["num_small_gt"] >= min_gt]

    return summary.sort_values(
        by=["recall", "avg_best_score", "num_small_gt"],
        ascending=[True, True, False],
    )


def compare_small_object_summaries(before_summary, after_summary, hard_class_ids=None):
    """Create before/after comparison table for class-level small-object performance."""
    before = before_summary.rename(
        columns={
            "num_matched": "before_matched",
            "recall": "before_recall",
            "avg_best_iou": "before_avg_iou",
            "avg_best_score": "before_avg_score",
        }
    )

    after = after_summary.rename(
        columns={
            "num_matched": "after_matched",
            "recall": "after_recall",
            "avg_best_iou": "after_avg_iou",
            "avg_best_score": "after_avg_score",
        }
    )

    compare_df = before.merge(
        after,
        on=["category_id", "category_name", "num_small_gt"],
        how="outer",
    )

    compare_df["delta_recall"] = compare_df["after_recall"] - compare_df["before_recall"]
    compare_df["delta_avg_score"] = compare_df["after_avg_score"] - compare_df["before_avg_score"]
    compare_df["delta_avg_iou"] = compare_df["after_avg_iou"] - compare_df["before_avg_iou"]

    if hard_class_ids is not None:
        compare_df = compare_df[compare_df["category_id"].isin(hard_class_ids)]

    return compare_df.sort_values(
        by=["delta_recall", "delta_avg_score"],
        ascending=[False, False],
    )


In [ ]:
def image_has_small_objects_from_hard_classes(coco, img_id, hard_class_ids, min_count=1):
    """Check whether an image has small objects from the selected hard classes."""
    stats = image_small_object_stats(coco, img_id)
    matched_anns = [
        ann
        for ann in stats["small"]
        if int(ann["category_id"]) in hard_class_ids
    ]

    return len(matched_anns) >= min_count, matched_anns


def mine_hard_images_from_coco(coco, hard_class_ids, exclude_img_ids=None, min_count=1):
    """Mine images from full COCO annotations containing small objects of hard classes."""
    exclude_img_ids = set(exclude_img_ids or [])
    candidate_rows = []

    for img_id in coco.imgs.keys():
        if img_id in exclude_img_ids:
            continue

        has_hard_small, matched_anns = image_has_small_objects_from_hard_classes(
            coco=coco,
            img_id=img_id,
            hard_class_ids=hard_class_ids,
            min_count=min_count,
        )

        if not has_hard_small:
            continue

        category_counts = defaultdict(int)
        for ann in matched_anns:
            category_counts[int(ann["category_id"])] += 1

        candidate_rows.append(
            {
                "image_id": img_id,
                "num_hard_small_objects": len(matched_anns),
                "category_counts": dict(category_counts),
            }
        )

    candidate_df = pd.DataFrame(candidate_rows)

    if len(candidate_df) == 0:
        return candidate_df

    return candidate_df.sort_values(
        by="num_hard_small_objects",
        ascending=False,
    )


def count_small_objects_by_class(coco, img_ids):
    """Count small objects per class for a set of COCO image ids."""
    counts = defaultdict(int)

    for img_id in img_ids:
        stats = image_small_object_stats(coco, img_id)
        for ann in stats["small"]:
            counts[int(ann["category_id"])] += 1

    rows = [
        {
            "category_id": cid,
            "category_name": get_cat_name(coco, cid),
            "num_small_objects": count,
        }
        for cid, count in counts.items()
    ]

    if not rows:
        return pd.DataFrame(columns=["category_id", "category_name", "num_small_objects"])

    return pd.DataFrame(rows).sort_values(
        by="num_small_objects",
        ascending=False,
    )


In [ ]:
def evaluate_rpn_recall_for_class(
    model,
    dataloader,
    coco,
    category_id,
    device=DEVICE,
    size_bucket="small",
    iou_threshold=0.5,
):
    """Evaluate RPN recall for one class and one size bucket."""
    model.eval()

    matched_total = 0
    gt_total = 0
    all_max_ious = []

    with torch.no_grad():
        for images, targets in dataloader:
            pipe = run_internal_frcnn_pipeline(model, images, targets, device=device)
            transformed_targets = pipe["transformed_targets"]
            proposals = pipe["proposals"]

            for i in range(len(images)):
                labels = targets[i]["labels"].to(device)
                areas = targets[i]["area"].to(device)

                class_mask = labels == category_id
                size_mask = get_size_mask(areas, size_bucket)
                mask = class_mask & size_mask

                gt_boxes = transformed_targets[i]["boxes"][mask]

                if len(gt_boxes) == 0:
                    continue

                recall, max_iou_per_gt = proposal_recall_at_iou(
                    gt_boxes,
                    proposals[i],
                    iou_threshold=iou_threshold,
                )

                matched_total += int((max_iou_per_gt >= iou_threshold).sum().item())
                gt_total += len(gt_boxes)
                all_max_ious.extend(max_iou_per_gt.detach().cpu().tolist())

    return {
        "category_id": category_id,
        "category_name": get_cat_name(coco, category_id),
        "size_bucket": size_bucket,
        "iou_threshold": iou_threshold,
        "matched": matched_total,
        "total": gt_total,
        "recall": matched_total / gt_total if gt_total > 0 else None,
        "mean_max_iou": float(np.mean(all_max_ious)) if all_max_ious else None,
    }


def get_class_gt_match_rows(df, category_id, size_bucket="small"):
    return df[
        (df["category_id"] == category_id)
        & (df["size_bucket"] == size_bucket)
    ].copy()


def run_and_show_predictions(
    model,
    dataset,
    img_id,
    device=DEVICE,
    score_threshold=0.3,
    class_id=None,
    max_boxes=30,
    categories=None,
):
    """Visualize GT and predictions for one image, optionally filtered to one class."""
    idx = dataset.img_ids.index(img_id)
    image_tensor, target = dataset[idx]

    model.eval()
    with torch.no_grad():
        output = model([image_tensor.to(device)])[0]

    pred_keep = output["scores"].detach().cpu() >= score_threshold

    if class_id is not None:
        gt_keep = target["labels"] == class_id
        pred_keep = pred_keep & (output["labels"].detach().cpu() == class_id)
    else:
        gt_keep = torch.ones(len(target["boxes"]), dtype=torch.bool)

    print("Predictions above threshold:", int(pred_keep.sum().item()))
    print("GT objects shown:", int(gt_keep.sum().item()))

    print("\nGround truth:")
    show_image_with_boxes(
        image_tensor,
        boxes=target["boxes"][gt_keep],
        labels=target["labels"][gt_keep],
        max_boxes=max_boxes,
        title="Ground truth",
        categories=categories,
    )

    print("\nPredictions:")
    show_image_with_boxes(
        image_tensor,
        boxes=output["boxes"].detach().cpu()[pred_keep],
        labels=output["labels"].detach().cpu()[pred_keep],
        scores=output["scores"].detach().cpu()[pred_keep],
        max_boxes=max_boxes,
        title=f"Predictions score >= {score_threshold}",
        categories=categories,
    )


## 2. Load COCO annotations and create a small-object debug subset


In [ ]:
train_coco = COCO(TRAIN_ANN)
val_coco = COCO(VAL_ANN)

print("Train images:", len(train_coco.imgs))
print("Train annotations:", len(train_coco.anns))
print("Val images:", len(val_coco.imgs))
print("Val annotations:", len(val_coco.anns))
print("Categories:", len(val_coco.cats))

print("\nFirst 10 categories:")
for cat in val_coco.loadCats(val_coco.getCatIds())[:10]:
    print(cat)


In [ ]:
val_small_object_img_ids = find_images_with_small_objects(
    val_coco,
    min_small_objects=MIN_SMALL_OBJECTS_PER_IMAGE,
)

print("Val images with at least", MIN_SMALL_OBJECTS_PER_IMAGE, "small objects:", len(val_small_object_img_ids))

debug_img_ids = random.sample(
    val_small_object_img_ids,
    min(NUM_DEBUG_IMAGES, len(val_small_object_img_ids)),
)

val_img_ids = debug_img_ids[:NUM_VAL]

print("Selected debug/val images:", len(val_img_ids))
print(val_img_ids[:10])


In [ ]:
download_image_subset(
    coco=val_coco,
    img_ids=val_img_ids,
    split="val2014",
    image_root=VAL_IMAGE_ROOT,
)

show_coco_image_with_gt(val_coco, val_img_ids[0], VAL_IMAGE_ROOT)


In [ ]:
val_dataset = CocoDetectionForFasterRCNN(
    coco=val_coco,
    img_ids=val_img_ids,
    image_root=VAL_IMAGE_ROOT,
)

val_loader = make_loader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

sample_image, sample_target = val_dataset[0]

print("image tensor shape:", sample_image.shape)
print("target keys:", sample_target.keys())
print("boxes shape:", sample_target["boxes"].shape)
print("labels shape:", sample_target["labels"].shape)
print("area shape:", sample_target["area"].shape)

print("\nFirst 5 boxes:")
print(sample_target["boxes"][:5])

print("\nFirst 5 labels:")
print(sample_target["labels"][:5])

print("\nFirst 5 areas and size buckets:")
for area in sample_target["area"][:5]:
    print(float(area), coco_size_bucket_from_area(float(area)))


## 3. Inspect pretrained Faster R-CNN and RPN behavior on real COCO images


In [ ]:
model, categories = make_fasterrcnn_model(
    use_small_anchors=False,
    freeze_backbone=False,
    apply_proposal_settings=False,
    device=DEVICE,
)
model.eval()

print("Number of model categories:", len(categories))
print(categories[:15])


In [ ]:
sample_image, sample_target = val_dataset[0]

with torch.no_grad():
    output = model([sample_image.to(DEVICE)])[0]

print("Output keys:", output.keys())
print("boxes:", output["boxes"].shape)
print("labels:", output["labels"].shape)
print("scores:", output["scores"].shape)

show_image_with_boxes(
    sample_image,
    boxes=output["boxes"],
    labels=output["labels"],
    scores=output["scores"],
    categories=categories,
    max_boxes=20,
    title="Pretrained Faster R-CNN predictions",
)


In [ ]:
images, targets = next(iter(val_loader))
pipe = run_internal_frcnn_pipeline(model, images, targets, device=DEVICE)

transformed_images = pipe["transformed_images"]
transformed_targets = pipe["transformed_targets"]
features = pipe["features"]
proposals = pipe["proposals"]

print("Original image shape:", images[0].shape)
print("Transformed batch shape:", transformed_images.tensors.shape)
print("Transformed image sizes:", transformed_images.image_sizes)

print("\nFeature maps:")
for name, feat in features.items():
    print(name, tuple(feat.shape))

print("\nRPN proposals:")
print(type(proposals), len(proposals))
print(proposals[0].shape)

print("\nTransformed target boxes:")
print(transformed_targets[0]["boxes"].shape)


In [ ]:
gt_boxes = transformed_targets[0]["boxes"]
proposal_boxes = proposals[0]

for thr in [0.3, 0.5, 0.7, 0.9]:
    recall, max_iou_per_gt = proposal_recall_at_iou(
        gt_boxes,
        proposal_boxes,
        iou_threshold=thr,
    )
    print(f"RPN proposal recall @ IoU {thr}: {recall:.3f}")

size_result = recall_by_size_bucket(
    original_target=targets[0],
    transformed_target=transformed_targets[0],
    proposals=proposals[0],
    iou_threshold=IOU_THRESH,
)

print(f"\nRPN recall by size bucket @ {IOU_THRESH}:")
for bucket, info in size_result.items():
    print(bucket, "num_gt:", info["num_gt"], "recall:", info["recall"])
    if info["max_iou_per_gt"] is not None:
        print("  max IoUs:", info["max_iou_per_gt"].numpy())


In [ ]:
pretrained_rpn_summary, pretrained_rpn_rows = evaluate_rpn_recall_dataset(
    model,
    val_loader,
    device=DEVICE,
    iou_threshold=IOU_THRESH,
)

print(f"Pretrained default Faster R-CNN RPN Recall @ {IOU_THRESH}")
for bucket, values in pretrained_rpn_summary.items():
    print(bucket, values)


## 4. Create small-object-biased training subset


In [ ]:
train_small_object_img_ids = find_images_with_small_objects(
    train_coco,
    min_small_objects=MIN_SMALL_OBJECTS_PER_IMAGE,
)

print("Train images with at least", MIN_SMALL_OBJECTS_PER_IMAGE, "small objects:", len(train_small_object_img_ids))

train_img_ids = random.sample(
    train_small_object_img_ids,
    min(NUM_TRAIN, len(train_small_object_img_ids)),
)

print("Train subset:", len(train_img_ids))
print("Val subset:", len(val_img_ids))


In [ ]:
download_image_subset(
    coco=train_coco,
    img_ids=train_img_ids,
    split="train2014",
    image_root=TRAIN_IMAGE_ROOT,
)

train_dataset = CocoDetectionForFasterRCNN(
    coco=train_coco,
    img_ids=train_img_ids,
    image_root=TRAIN_IMAGE_ROOT,
)

train_loader = make_loader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


## 5. Fine-tune default-anchor Faster R-CNN


In [ ]:
model_default, _ = make_fasterrcnn_model(
    use_small_anchors=False,
    freeze_backbone=True,
    apply_proposal_settings=False,
    device=DEVICE,
)

optimizer_default = make_optimizer(model_default)
scaler_default = make_scaler()

trainable = sum(p.numel() for p in model_default.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model_default.parameters() if not p.requires_grad)

print("Default-anchor model")
print("Trainable params:", trainable)
print("Frozen params:", frozen)


In [ ]:
for epoch in range(NUM_EPOCHS):
    avg_loss = train_one_epoch(
        model=model_default,
        loader=train_loader,
        optimizer=optimizer_default,
        scaler=scaler_default,
        device=DEVICE,
        epoch=epoch,
        print_every=25,
    )

    print(f"Default-anchor epoch {epoch} average loss: {avg_loss:.4f}")

    summary, _ = evaluate_rpn_recall_dataset(
        model_default,
        val_loader,
        device=DEVICE,
        iou_threshold=IOU_THRESH,
    )

    print(f"Default-anchor validation RPN Recall @ {IOU_THRESH} after epoch {epoch}")
    for bucket, values in summary.items():
        print(bucket, values)


In [ ]:
apply_rpn_proposal_settings(model_default)

default_more_props_summary, _ = evaluate_rpn_recall_dataset(
    model_default,
    val_loader,
    device=DEVICE,
    iou_threshold=IOU_THRESH,
)

print(f"Default-anchor RPN Recall @ {IOU_THRESH} with higher proposal counts")
for bucket, values in default_more_props_summary.items():
    print(bucket, values)


## 6. Fine-tune small-anchor Faster R-CNN


In [ ]:
model_small_anchors, _ = make_fasterrcnn_model(
    use_small_anchors=True,
    freeze_backbone=True,
    apply_proposal_settings=True,
    device=DEVICE,
)

small_anchor_pretrain_summary, _ = evaluate_rpn_recall_dataset(
    model_small_anchors,
    val_loader,
    device=DEVICE,
    iou_threshold=IOU_THRESH,
)

print(f"Small-anchor model RPN Recall @ {IOU_THRESH} before fine-tuning")
for bucket, values in small_anchor_pretrain_summary.items():
    print(bucket, values)


In [ ]:
optimizer_small = make_optimizer(model_small_anchors)
scaler_small = make_scaler()

for epoch in range(NUM_EPOCHS):
    avg_loss = train_one_epoch(
        model=model_small_anchors,
        loader=train_loader,
        optimizer=optimizer_small,
        scaler=scaler_small,
        device=DEVICE,
        epoch=epoch,
        print_every=25,
    )

    print(f"Small-anchor epoch {epoch} average loss: {avg_loss:.4f}")

    summary, _ = evaluate_rpn_recall_dataset(
        model_small_anchors,
        val_loader,
        device=DEVICE,
        iou_threshold=IOU_THRESH,
    )

    print(f"Small-anchor validation RPN Recall @ {IOU_THRESH} after epoch {epoch}")
    for bucket, values in summary.items():
        print(bucket, values)


In [ ]:
run_and_show_predictions(
    model=model_small_anchors,
    dataset=val_dataset,
    img_id=val_img_ids[0],
    device=DEVICE,
    score_threshold=0.3,
    categories=categories,
)


## 7. Per-class small-object evaluation


In [ ]:
baseline_df = evaluate_gt_centered_per_class(
    model=model_small_anchors,
    dataloader=val_loader,
    coco=val_coco,
    device=DEVICE,
    iou_threshold=IOU_THRESH,
    score_threshold=SCORE_THRESH,
)

baseline_small_summary = summarize_small_object_class_performance(
    baseline_df,
    min_gt=MIN_SMALL_GT,
)

baseline_small_summary


In [ ]:
missed_small_df = baseline_df[
    (baseline_df["size_bucket"] == "small")
    & (baseline_df["matched"] == False)
].copy()

missed_class_counts = (
    missed_small_df.groupby(["category_id", "category_name"])
    .size()
    .reset_index(name="num_missed_small")
    .sort_values("num_missed_small", ascending=False)
)

missed_class_counts.head(20)


In [ ]:
hard_classes_df = baseline_small_summary[
    (baseline_small_summary["recall"] < POOR_RECALL_THRESHOLD)
    & (baseline_small_summary["num_small_gt"] >= MIN_SMALL_GT)
].copy()

hard_class_ids = set(hard_classes_df["category_id"].astype(int).tolist())

print("Hard class IDs:", hard_class_ids)
print("Hard classes:")
for cid in sorted(hard_class_ids):
    print(cid, get_cat_name(val_coco, cid))

hard_classes_df


## 8. Mine additional hard-class small-object images from full COCO train


In [ ]:
candidate_df = mine_hard_images_from_coco(
    coco=train_coco,
    hard_class_ids=hard_class_ids,
    exclude_img_ids=train_img_ids,
    min_count=1,
)

print("Candidate images from full COCO train:", len(candidate_df))
candidate_df.head(20)


In [ ]:
added_hard_img_ids = (
    candidate_df["image_id"]
    .astype(int)
    .tolist()[:NUM_ADDED_HARD_IMAGES]
)

expanded_train_img_ids = list(train_img_ids) + added_hard_img_ids

print("Original train size:", len(train_img_ids))
print("Added hard-mined images:", len(added_hard_img_ids))
print("Expanded train size:", len(expanded_train_img_ids))
print("First added image ids:", added_hard_img_ids[:10])


In [ ]:
download_image_subset(
    coco=train_coco,
    img_ids=added_hard_img_ids,
    split="train2014",
    image_root=TRAIN_IMAGE_ROOT,
)

expanded_train_dataset = CocoDetectionForFasterRCNN(
    coco=train_coco,
    img_ids=expanded_train_img_ids,
    image_root=TRAIN_IMAGE_ROOT,
)

expanded_train_loader = make_loader(
    expanded_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

print("Expanded train batches:", len(expanded_train_loader))


In [ ]:
original_small_counts = count_small_objects_by_class(train_coco, train_img_ids)
expanded_small_counts = count_small_objects_by_class(train_coco, expanded_train_img_ids)

hard_original_counts = original_small_counts[
    original_small_counts["category_id"].isin(hard_class_ids)
]

hard_expanded_counts = expanded_small_counts[
    expanded_small_counts["category_id"].isin(hard_class_ids)
]

print("Original hard-class small object counts:")
display(hard_original_counts)

print("Expanded hard-class small object counts:")
display(hard_expanded_counts)


## 9. Train the expanded-data model and compare against baseline


In [ ]:
model_expanded, _ = make_fasterrcnn_model(
    use_small_anchors=True,
    freeze_backbone=True,
    apply_proposal_settings=True,
    device=DEVICE,
)

optimizer_expanded = make_optimizer(model_expanded)
scaler_expanded = make_scaler()

print("Expanded-data model ready.")


In [ ]:
# Option A: same number of epochs as the baseline.
# This changes both data distribution and number of optimizer steps because the dataset is larger.

# for epoch in range(NUM_EPOCHS):
#     avg_loss = train_one_epoch(
#         model=model_expanded,
#         loader=expanded_train_loader,
#         optimizer=optimizer_expanded,
#         scaler=scaler_expanded,
#         device=DEVICE,
#         epoch=epoch,
#         print_every=25,
#     )

#     print(f"Expanded-data epoch {epoch} average loss: {avg_loss:.4f}")


In [ ]:
# Option B: uncomment this instead of the epoch loop above if you want a cleaner fixed-step comparison.
BASELINE_STEPS = len(train_loader) * NUM_EPOCHS
avg_loss = train_fixed_steps(
    model=model_expanded,
    loader=expanded_train_loader,
    optimizer=optimizer_expanded,
    scaler=scaler_expanded,
    device=DEVICE,
    max_steps=BASELINE_STEPS,
    print_every=25,
)
print("Expanded-data fixed-step average loss:", avg_loss)


In [ ]:
expanded_after_df = evaluate_gt_centered_per_class(
    model=model_expanded,
    dataloader=val_loader,
    coco=val_coco,
    device=DEVICE,
    iou_threshold=IOU_THRESH,
    score_threshold=SCORE_THRESH,
)

expanded_after_small_summary = summarize_small_object_class_performance(
    expanded_after_df,
    min_gt=MIN_SMALL_GT,
)

expanded_hard_compare_df = compare_small_object_summaries(
    before_summary=baseline_small_summary,
    after_summary=expanded_after_small_summary,
    hard_class_ids=hard_class_ids,
)

expanded_hard_compare_df[
    [
        "category_id",
        "category_name",
        "num_small_gt",
        "before_matched",
        "after_matched",
        "before_recall",
        "after_recall",
        "delta_recall",
        "before_avg_iou",
        "after_avg_iou",
        "delta_avg_iou",
        "before_avg_score",
        "after_avg_score",
        "delta_avg_score",
    ]
]


## 10. Class-level debugging and visualization


In [ ]:
suspect_rpn_before = evaluate_rpn_recall_for_class(
    model=model_small_anchors,
    dataloader=val_loader,
    coco=val_coco,
    category_id=SUSPECT_ID,
    device=DEVICE,
    size_bucket="small",
    iou_threshold=IOU_THRESH,
)

suspect_rpn_after = evaluate_rpn_recall_for_class(
    model=model_expanded,
    dataloader=val_loader,
    coco=val_coco,
    category_id=SUSPECT_ID,
    device=DEVICE,
    size_bucket="small",
    iou_threshold=IOU_THRESH,
)

print("Before:", suspect_rpn_before)
print("After:", suspect_rpn_after)


In [ ]:
suspect_before_rows = get_class_gt_match_rows(
    baseline_df,
    category_id=SUSPECT_ID,
    size_bucket="small",
)

suspect_after_rows = get_class_gt_match_rows(
    expanded_after_df,
    category_id=SUSPECT_ID,
    size_bucket="small",
)

suspect_compare = suspect_before_rows.merge(
    suspect_after_rows,
    on=["image_id", "gt_idx", "category_id", "category_name", "size_bucket"],
    suffixes=("_before", "_after"),
)

suspect_compare["changed"] = (
    suspect_compare["matched_before"] != suspect_compare["matched_after"]
)

suspect_compare[
    [
        "image_id",
        "gt_idx",
        "matched_before",
        "matched_after",
        "best_iou_before",
        "best_iou_after",
        "best_score_before",
        "best_score_after",
        "reason_before",
        "reason_after",
        "changed",
    ]
].sort_values("changed", ascending=False)


In [ ]:
# Change img_id/class_id as needed.
run_and_show_predictions(
    model=model_small_anchors,
    dataset=val_dataset,
    img_id=val_img_ids[0],
    device=DEVICE,
    score_threshold=0.3,
    class_id=SUSPECT_ID,
    categories=categories,
)

run_and_show_predictions(
    model=model_expanded,
    dataset=val_dataset,
    img_id=val_img_ids[0],
    device=DEVICE,
    score_threshold=0.3,
    class_id=SUSPECT_ID,
    categories=categories,
)
